# UKFによる状態推定


1. UKFによる状態推定
2. UKFによる外乱推定
3. UKFによるパラメータ推定
4. UKFによる外乱・パラメータの同時推定

## 運動方程式

以下の剛性が非線形なバネ-マス-ダンパ系の運動方程式とする。

$$
m \ddot{x} +  c \dot{x} + k x |x| =  u + d
$$

- $m$ : 慣性 [$kg$]
- $c$ : 粘性 [$Ns/m$]
- $k$ : 剛性 [$N/m$]
- $u$ : 既知の制御入力 [$N$]
- $d$ : 未知外乱 [$N$]

## 状態方程式

状態ベクトル$X$とする。

$$
X = \begin{bmatrix} x \\ v \end{bmatrix}
$$

これより、連続時間の状態方程式は以下となる。

$$
\begin{bmatrix}
\dot{x} \\ \dot{v}
\end{bmatrix} =
\begin{bmatrix}
v \\
\frac{1}{m}\left( u - c v - k x|x| + d \right)
\end{bmatrix}
$$

前進Euler法による離散時間モデルは、以下となる。
$$
X_{n+1} = f(X_n + u_n) + w_n\\ \downarrow \\
\begin{bmatrix}
x_{n+1} \\ v_{n+1}
\end{bmatrix} =
\begin{bmatrix}
x_{n} + \Delta t v_n \\
v_n + \frac{\Delta t}{m}\left( u_n - c v_n - k x_n|x_n| + d_n \right)
\end{bmatrix} + w_n
$$

$w_n$ は $\mathcal{N}(0, Q_n)$ に従うプロセスノイズである。


## 観測方程式

状態推定を行うため、観測値は速度とする。

$$
z_{n+1} = h(x_{n+1}) + v_{n+1} \\ \downarrow \\
z_{n+1} = v_{n+1} + v_{n+1}
$$

$v_{n+1}$は $\mathcal{N}(0, R_{n+1})$ に従う観測ノイズである。


## 1. 状態推定

状態推定では未知外乱は$d=0$として行う。

状態数は$n=2$である。

シグマポイントに関するパラメータは$\alpha=1, \kappa = 0, \beta = 2$とする。

$$
\lambda = \alpha^2(n+\kappa) -n
$$

平均計算用重み

$$
W_0^{(m)} = \frac{\lambda}{n+\lambda} , \quad W_i^{(m)} = \frac{1}{2(n+\lambda)}, \space i=1,...,4
$$

共分散計算用重み

$$
W_0^{(c)} = W_0^{(m)} + (1 - \alpha^2 + \beta) , \quad W_i^{(c)} = W_i^{(m)}, \space i=1,...,4
$$

以上を用いて、1ループの処理は以下となる。

#### 1.1. シグマポイントの計算

時刻$k$時の事後推定の推定状態を$\hat{x}_{k|k} \in \mathcal{R}^{2 \times 1}$、誤差共分散行列を$P_{k|k} \in \mathcal{R}^{2 \times 2}$とし、シグマポイント $\chi_{k|k} \in \mathcal{R}^{2 \times 1}$ を計算する。

$$
\begin{aligned}
\chi_{k|k}^{(0)} &= \hat{x}_{k|k} \\
\chi_{k|k}^{(i)} &= \hat{x}_{k|k} + \sqrt{n+\lambda} \left(\sqrt{P_{{k|k}}}\right)_i, \space i = 1,2 \\
\chi_{k|k}^{(n+i)} &= \hat{x}_{k|k} - \sqrt{n+\lambda} \left(\sqrt{P_{{k|k}}}\right)_i, \space i = 1,2 \\
\end{aligned}
$$

#### 1.2. 次状態の予測値(事前推定値)を計算

シグマポイント$\chi_{k|k}$を状態方程式を通過させ、次状態のシグマポイント$\chi_{k+1|k}$を求める。

$$
\chi_{k+1|k}^{(i)} = f(\chi_{k|k}^{(i)}, u_k), \space i=0,..,4
$$

$\chi_{k+1|k}$の平均値を計算し、次状態の予測値を求める。

$$
\hat{x}_{k+1|k} = \sum_{i=0}^{4}W_i^{(m)}\chi_{k+1|k}^{(i)}
$$

次状態の予測誤差共分散を求める。

$$
P_{k+1|k} = \sum_{i=0}^{4} W_i^{(c)} \left( \chi_{k+1|k}^{(i)} - \hat{x}_{k+1|k} \right) \left( \chi_{k+1|k}^{(i)} - \hat{x}_{k+1|k} \right)^T + Q_k
$$

#### 1.3 事前推定値からシグマポイントを計算

事前推定値の$\hat{x}_{k+1|k}$と$P_{k+1|k}$からシグマポイント$\psi_{k+1|k}  \in \mathcal{R}^{2 \times 1}$を求める。

$$
\begin{aligned}
\psi_{k+1|k}^{(0)} &= \hat{x}_{k+1|k} \\
\psi_{k+1|k}^{(i)} &= \hat{x}_{k+1|k} + \sqrt{n+\lambda} \left(\sqrt{P_{{k+1|k}}}\right)_i, \space i = 1,2 \\
\psi_{k+1|k}^{(n+i)} &= \hat{x}_{k+1|k} - \sqrt{n+\lambda} \left(\sqrt{P_{{k+1|k}}}\right)_i, \space i = 1,2 \\
\end{aligned}
$$

#### 1.4 観測値の予測値を計算

シグマポイント$\psi_{k+1|k}$を観測方程式を通過させ、観測空間でのシグマポイント$Z_{k+1} \in \mathcal{R}^{1 \times 1}$を求める。

$$
Z_{k+1}^{(i)} = h(\psi_{k+1|k}^{(i)}) , \space i=0,...,4
$$

$Z_{k+1}$の平均値を計算し、観測値の予測値$\hat{z}_{k+1|k}  \in \mathcal{R}^{1 \times 1}$を求める。

$$
\hat{z}_{k+1|k} = \sum_0^{4} W_i^{(m)} Z_{k+1}^{(i)}
$$

観測値の予測誤差共分散$S_{k+1} \in \mathcal{R}^{1 \times 1}$を求める。

$$
S_{k+1} = \sum_{i=0}^{4} W_i^{(c)} \left( Z_{k+1}^{(i)} - \hat{z}_{k+1|k} \right) \left( Z_{k+1}^{(i)} - \hat{z}_{k+1|k} \right)^T + R_{k+1}
$$

#### 1.5 交差共分散を計算

シグマポイント$\psi_{k+1|k}$, Z_{k+1}$から交差共分散$P_{xz} \in \mathcal{R}^{2 \times 1}$ を計算する。

$$
P_{xz} = \sum_{i=0} W_i^{(c)} \left( \psi_{k+1}^{(i)} - \hat{x}_{k+1|k} \right) \left( Z_{k+1}^{(i)} - \hat{z}_{k+1|k} \right)^T
$$

#### 1.6 Kalma Gainを計算

観測値の予測誤差共分散$S_{k+1}$ と交差共分散 $P_{xz}$からKalman Gain $K_{k+1} \in \mathcal{R}^{2 \times 1}$ を計算する。

$$
K_{k+1} = P_{xz}S_{k+1}^{-1}
$$

#### 1.7 状態の更新

システムの測定値$z_{k+1} \in \mathcal{R}^{1 \times 1}$ と観測値の予測値$\hat{z}_{k+1|k}$からinnovation$\tilde{z}_{k+1} \in \mathcal{R}^{1 \times 1}$を求める。

$$
\tilde{z}_{k+1} = z_{k+1} - \tilde{z}_{k+1|k}
$$

時刻$k+1$時の事後推定の状態$\hat{x}_{k+1|k+1} \in \mathcal{R}^{2 \times 1}$ を事前推定の状態$\hat{x}_{k+1|k}$とKalman Gain $K_{k+1}$により以下で更新する。

$$
\hat{x}_{k+1|k+1} = \hat{x}_{k+1|k} + K_{k+1}
$$

#### 1.8 誤差共分散の更新

時刻$k+1$時の事後推定の誤差共分散の$P_{k+1|k+1} \in \mathcal{R}^{2 \times 2}$ を事前状態の誤差共分散$P_{k+1|k}$、Kalman Gain$K_{k+1}$と観測値の予測誤差共分散$S_{k+1}$を持ちて以下で更新する。

$$
P_{k+1|k+1} = P_{k+1|k} - K_{k+1} S_{k+1} K_{k+1}^T
$$

これで時刻$k+1$での事後推定値が求まる。以降は1.1に戻り、各時刻での推定を行う。